# Compute cost: rollout and update time / peak VRAM vs sequence length

Same paper style as `main_figure_vis.ipynb` (fonts, fixed palette and legend order, MATE drawn thicker on top,
dashed grid, no top/right spines). One flat full-column 2x2 figure (5.5 in text width, each plot area about 3.1 : 1):

| | left | right |
|---|---|---|
| **top** | Rollout time | Update time |
| **bottom** | Rollout peak GPU memory | Update peak GPU memory |

Column titles on the top row only, x label on the bottom row only, y label (one unit per row) on the left
column only; all four plot areas are identical. The legend is one row below the grid, in the style of
`figures/main/legend.pdf` (`LEGEND_MODE = "bottom"`); a flat panel has no room for it inside. Rollout VRAM is **cut at the top** (`YCLIP`): GPT-2's KV cache grows with `L` while the recurrent
baselines stay flat near 0, so GPT-2 leaves the panel and its value at the longest `L` is printed in the corner. Include the PDF at native size (no `width=`).

Data: `rollout_time.txt` / `update_time.txt` from `check_time.sh` (`check_time.py`), batch 64, hidden 128,
1 layer, MATE with the `gpt_ffn` embedder. Each point is the mean over 10 timed trials after 3 warm-up runs.
The std is narrower than the lines everywhere, so it is not drawn; the "Std bound" section gives a rounding-aware
upper bound on std / mean for the caption.
- **Rollout**: `L` consecutive single-step calls under `no_grad` (eval mode), i.e. one length-`L` rollout.
- **Update**: one forward + backward of the sequence model on a `(L, 64)` batch with a dummy loss.

In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter, FuncFormatter, MaxNLocator, MultipleLocator

try:
    display
except NameError:            # plain-python execution (smoke test); Jupyter defines display()
    display = print


from paper_style import set_paper_style   # shared style: font sizes live in paper_style.py only


set_paper_style()

## Config

`METHODS` / `STYLE` are the main-figure palette and order without Markov (no memory, nothing to time).
`PHASES` maps each panel to its log; `METRICS` converts the log units (s, MB) to the plotted ones.

In [2]:
METHODS = ["MATE", "GPT-2", "LSTM", "SplAgger", "Mamba"]    # legend / draw order (fixed paper order; no Markov here)
STYLE = {                            # fixed paper palette (same as main_figure_vis.ipynb)
    "MATE":     dict(color="#E41A1C", ls="-"),   # ours: drawn thicker and on top (EMPHASIS)
    "GPT-2":    dict(color="#009E73", ls="-"),
    "LSTM":     dict(color="#0072B2", ls="-"),
    "SplAgger": dict(color="#AA3377", ls="-"),
    "Mamba":    dict(color="#E69F00", ls="-"),
}
LOG_NAME = {"MATE": "mate", "GPT-2": "gpt", "LSTM": "lstm", "SplAgger": "splagger", "Mamba": "mamba"}   # 'Model:' in the logs

HIDDEN_SIZE = 128
PHASES = {                           # column key -> log file + column title; order = left, right
    "rollout": dict(path="./rollout_time.txt", title="Rollout"),
    "update":  dict(path="./update_time.txt", title="Update"),
}
METRICS = {                          # row key -> columns, unit conversion from the log units (s, MB), y label; order = top, bottom
    "time": dict(col="Time", err="Time_std", scale=1e3, ylabel="Time (ms)"),
    "vram": dict(col="VRAM", err="VRAM_std", scale=1 / 1024, ylabel="Memory (GB)"),
}
XLABEL = "Sequence Length"           # L: steps per rollout / length of the update batch (domain-neutral, not "episode")
XMAX = 1000                          # x axis 0..XMAX, last tick labelled exactly XMAX
SHOW_STD = False                     # std is < the line width everywhere: state its bound in the caption instead
N_TRIALS = 10                        # timed runs per point in check_time.py (np.std, ddof=0)
LOG_HALF_UNIT = {"time": 0.5e-4, "vram": 0.5e-2}   # half of the last logged digit: s with 4, MB with 2 decimals
YCLIP = {                            # (metric, phase) -> y limit that cuts the top off an outlier curve
    ("vram", "rollout"): dict(          # GPT-2's KV cache grows with L while the recurrent baselines stay flat:
        top=0.08,                        # keep its rise visible, let it leave the panel, and print its end value
        note=("GPT-2", "GPT-2: {y:.2f} GB at L = {L:d}")),   # (method, text) drawn at the top right; None -> no note
}
LEGEND_MODE = "bottom"               # "bottom": one row below the grid (user's choice) | "top": above it | "panel": inside LEGEND_CELL
LEGEND_TOP = dict(handlelength=1.8, handletextpad=0.4, columnspacing=0.8, borderpad=0.35)   # as the T-Maze training-time panel
LEGEND_GAP = 0.04                    # space between the legend row and the x labels (bottom) / column titles (top) (in)
LEGEND_CELL = ("vram", "rollout")    # LEGEND_MODE="panel" only: rollout VRAM has room above its flat lines (needs PLOT_H >~ 1)
LEGEND_IN_PANEL = dict(loc="center right", ncol=2)   # ax.legend kwargs; y limit is raised only if it would cover data

# 2x2 grid in ONE figure: rows = METRICS, columns = PHASES. Compact: column titles on the top row only,
# x tick labels + x label on the bottom row only, y label on the left column only (one unit per row).
FIG_WIDTH = 5.5                      # ICLR \textwidth (in); include at native size
PLOT_H = 0.75                        # height of each plot area (in): flat full-column figure, each panel about 3.1 : 1;
                                     # must stay >= the longest y label ("Memory (GB)" = 0.72 in at 9 pt)
COL_GAP = 0.08                       # extra space between the columns (in), on top of the right column's y tick labels
ROW_GAP = 0.05                       # extra space between the rows (in), on top of the overhanging y tick labels
YTICK_RESERVE = "0000"
XTICK_RESERVE_RIGHT = "1000"
OUTER_PAD = 0.02
LINE_W = 0.9
MARKER, MARKER_SIZE, MARKER_EDGE = "o", 2.6, 0.7   # a dot on every measured L (as the T-Maze training-time panel); None -> lines only
EMPHASIS, EMPHASIS_SCALE = "MATE", 1.35
BAND_ALPHA = 0.15
LEGEND_EDGE = "#d9d9d9"
LEGEND_SHADOW = dict(size=1.5, alpha=0.35, layers=4)

FIG_DIR = Path("figures") / "timing"
OUT_STEM = "compute_cost"            # -> figures/timing/compute_cost.{pdf,png}

## Parse the logs

Parsed by key, so extra fields don't shift anything. The logs are appended to, so a repeated
(model, length) row is replaced by the newest one (with a warning).

In [3]:
def _parse_mean_std(raw, unit):
    """'0.0026 +- 0.0001 s' -> (0.0026, 0.0001)"""
    value = raw.replace(unit, "").strip()
    if "+-" in value:
        mean, std = value.split("+-")
        return float(mean.strip()), float(std.strip())
    return float(value), 0.0


def parse_log_file(filepath):
    """Parse a check_time.py log.

    Lines are 'Key: value | Key: value | ...'. Parsed by key rather than by
    column so extra fields (e.g. 'Trans') don't shift anything.
    """
    if not os.path.exists(filepath):
        warnings.warn(f"file not found: {filepath}")
        return pd.DataFrame()

    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            fields = {}
            for part in line.split("|"):
                if ":" not in part:
                    continue
                key, value = part.split(":", 1)
                fields[key.strip()] = value.strip()

            # 'Rollout time' in rollout_time.txt, 'Update time' in update_time.txt
            time_key = next((k for k in fields if k.endswith("time")), None)
            if time_key is None or "Peak VRAM" not in fields:
                continue
            try:
                time_mean, time_std = _parse_mean_std(fields[time_key], "s")
                vram_mean, vram_std = _parse_mean_std(fields["Peak VRAM"], "MB")
                data.append({
                    "Model": fields["Model"].lower(),
                    "Hidden": int(fields["Hidden"]),
                    "Layer": int(fields.get("Layer", 0)),
                    "Batch": int(fields.get("Batch", 0)),
                    "Seq": int(fields["Seq"]),
                    "Params": int(fields.get("Params", 0)),
                    "Time": time_mean, "Time_std": time_std,
                    "VRAM": vram_mean, "VRAM_std": vram_std,
                })
            except Exception as e:
                warnings.warn(f"could not parse line: {line!r} ({e})")
    return pd.DataFrame(data)


def load_phase(path, hidden_size=HIDDEN_SIZE):
    df = parse_log_file(path)
    if df.empty:
        return df
    df = df[df["Hidden"] == hidden_size]
    dup = df.duplicated(["Model", "Seq"], keep="last")
    if dup.any():                        # the logs are appended to: a re-run wins
        warnings.warn(f"{path}: {int(dup.sum())} duplicate (model, length) rows, keeping the last one")
        df = df[~dup]
    unknown = sorted(set(df["Model"]) - set(LOG_NAME.values()))
    if unknown:
        warnings.warn(f"{path}: models not in METHODS, not drawn: {unknown}")
    return df.sort_values(["Model", "Seq"]).reset_index(drop=True)


timing = {key: load_phase(p["path"]) for key, p in PHASES.items()}
for key, df in timing.items():
    print(f"{key}: {len(df)} rows, models {sorted(df['Model'].unique())}, lengths {sorted(df['Seq'].unique())}")

rollout: 50 rows, models ['gpt', 'lstm', 'mamba', 'mate', 'splagger'], lengths [np.int64(100), np.int64(200), np.int64(300), np.int64(400), np.int64(500), np.int64(600), np.int64(700), np.int64(800), np.int64(900), np.int64(1000)]
update: 50 rows, models ['gpt', 'lstm', 'mamba', 'mate', 'splagger'], lengths [np.int64(100), np.int64(200), np.int64(300), np.int64(400), np.int64(500), np.int64(600), np.int64(700), np.int64(800), np.int64(900), np.int64(1000)]


## Figure

`grid_layout()` places the four axes in inches (fixed tick-label reserves, as in the other figure notebooks'
`panel_margins`). The x axis runs from 0 to `XMAX` with the last tick labelled; y starts at 0; the legend
panel's y limit is raised only if the legend would cover a curve. File: `figures/timing/compute_cost.pdf`
(+ PNG preview).

In [4]:
def budget_tick_candidates(xmax):
    """Tick sets 0..xmax whose LAST tick is exactly xmax, best first: a round step dividing xmax (densest
    first), then a round step whose last tick is replaced by / followed by xmax, then [0, xmax/2, xmax],
    then [0, xmax]."""
    decade = 10.0 ** np.floor(np.log10(xmax))
    steps = [m * decade for m in (0.1, 0.2, 0.25, 0.5, 1, 2, 2.5, 5)]
    divides = [abs(xmax / s - round(xmax / s)) < 1e-6 for s in steps]
    for step, exact in zip(steps, divides):
        if exact and 3 <= round(xmax / step) <= 6:
            yield np.linspace(0, xmax, int(round(xmax / step)) + 1)
    for step, exact in zip(steps, divides):
        n = int(np.floor(xmax / step + 1e-9))
        if not exact and 2 <= n <= 6:
            ticks = list(np.arange(n + 1) * step)
            if xmax - ticks[-1] < 0.5 * step:
                ticks[-1] = xmax                               # too close to the end to label both
            else:
                ticks.append(xmax)
            yield np.array(ticks)
    yield np.array([0, xmax / 2, xmax])
    yield np.array([0, xmax])


def set_budget_xticks(ax, xmax, min_gap_pt=1.0):
    """x axis 0..xmax with the densest candidate tick set whose labels do not collide (call after the
    labels/title are set: it draws the figure to measure the tick labels). Plain numbers: a sequence
    length reads "1000", not "1k"."""
    ax.set_xlim(0, xmax)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _pos: f"{v:g}"))
    fig = ax.figure
    gap = min_gap_pt * fig.dpi / 72
    for ticks in budget_tick_candidates(xmax):
        ax.set_xticks(ticks)
        fig.canvas.draw()
        boxes = sorted((t.get_window_extent() for t in ax.get_xticklabels() if t.get_text()), key=lambda b: b.x0)
        if all(a.x1 + gap <= b.x0 for a, b in zip(boxes, boxes[1:])):
            break
    ax.set_xlim(0, xmax)
    return ticks


def style_legend_frame(legend):
    """Thin light edge plus a soft drop shadow (stacked offset copies with decreasing alpha; stays vector in PDF)."""
    frame = legend.get_frame()
    frame.set_linewidth(0.4)
    if LEGEND_SHADOW:
        n, size, alpha = LEGEND_SHADOW["layers"], LEGEND_SHADOW["size"], LEGEND_SHADOW["alpha"]
        effects = [pe.SimplePatchShadow(offset=(size * k / n, -size * k / n), shadow_rgbFace="black",
                                        alpha=alpha / n) for k in range(n, 0, -1)]
        frame.set_path_effects(effects + [pe.Normal()])


def _text_extent(s, size, rotation=0, weight="normal"):
    """(width, height) in inches of `s` at `size` pt with the current rcParams (0 for an empty string)."""
    if not s:
        return 0.0, 0.0
    fig = plt.figure()
    t = fig.text(0, 0, s, fontsize=size, rotation=rotation, fontweight=weight)
    fig.canvas.draw()
    bb = t.get_window_extent()
    plt.close(fig)
    return bb.width / fig.dpi, bb.height / fig.dpi


def top_legend_kwargs():
    return dict(loc="upper center", ncol=len(METHODS), frameon=True, fancybox=False, edgecolor=LEGEND_EDGE,
                facecolor="white", framealpha=1.0, borderaxespad=0, **LEGEND_TOP)


def top_legend_height():
    """Height (in) of the one-row legend plus its drop shadow, measured on a scratch figure."""
    fig = plt.figure()
    legend = fig.legend(handles=legend_handles(), **top_legend_kwargs())
    fig.canvas.draw()
    height = legend.get_window_extent().height / fig.dpi
    plt.close(fig)
    return height + (LEGEND_SHADOW["size"] / 72 if LEGEND_SHADOW else 0)


def grid_layout():
    """Axes rectangles (inches) of the 2x2 grid: equal plot areas, decorations paid once per row / column.
    Tick-label room is a fixed reserve (YTICK_RESERVE, XTICK_RESERVE_RIGHT), as in the other figure notebooks."""
    rc, pt = matplotlib.rcParams, 1 / 72
    ytick_w, ytick_h = _text_extent(YTICK_RESERVE, rc["ytick.labelsize"])
    ytick_room = ytick_w + (rc["ytick.major.size"] + rc["ytick.major.pad"]) * pt
    ylabel_w = max(_text_extent(METRICS[m]["ylabel"], rc["axes.labelsize"], rotation=90)[0] for m in METRICS)
    xtick_room = _text_extent("0", rc["xtick.labelsize"])[1] + (rc["xtick.major.size"] + rc["xtick.major.pad"]) * pt
    xlabel_h = _text_extent(XLABEL, rc["axes.labelsize"])[1] + rc["axes.labelpad"] * pt
    title_h = _text_extent("Ag", rc["axes.titlesize"], weight=rc["axes.titleweight"])[1] + rc["axes.titlepad"] * pt

    left = OUTER_PAD + ylabel_w + rc["axes.labelpad"] * pt + ytick_room
    right = OUTER_PAD + _text_extent(XTICK_RESERVE_RIGHT, rc["xtick.labelsize"])[0] / 2
    top = OUTER_PAD + max(title_h, ytick_h / 2)
    bottom = OUTER_PAD + xtick_room + xlabel_h
    if LEGEND_MODE == "top":
        top += top_legend_height() + LEGEND_GAP
    elif LEGEND_MODE == "bottom":
        bottom += top_legend_height() + LEGEND_GAP
    col_gap = COL_GAP + ytick_room                      # the right column keeps its own y tick labels
    row_gap = ROW_GAP + ytick_h                         # top y label of the lower row + bottom one of the upper row
    n_cols, n_rows = len(PHASES), len(METRICS)
    pw = (FIG_WIDTH - left - right - (n_cols - 1) * col_gap) / n_cols
    height = bottom + n_rows * PLOT_H + (n_rows - 1) * row_gap + top
    too_long = [m for m in METRICS if _text_extent(METRICS[m]["ylabel"], rc["axes.labelsize"])[0] > PLOT_H]
    if too_long:
        warnings.warn(f"y label longer than PLOT_H={PLOT_H} in for {too_long}: shorten it or raise PLOT_H")
    rects = {}
    for r, metric in enumerate(METRICS):
        y0 = bottom + (n_rows - 1 - r) * (PLOT_H + row_gap)
        for c, phase in enumerate(PHASES):
            rects[(metric, phase)] = (left + c * (pw + col_gap), y0, pw, PLOT_H)
    return (FIG_WIDTH, height), rects


def check_fits(fig):
    """Warn when a label is larger than its reserve (it would be clipped at the file edge)."""
    fig.canvas.draw()
    bb, fb = fig.get_tightbbox(fig.canvas.get_renderer()), fig.bbox_inches
    over = {"left": -bb.x0, "right": bb.x1 - fb.x1, "bottom": -bb.y0, "top": bb.y1 - fb.y1}
    over = {k: round(v, 3) for k, v in over.items() if v > 0.005}
    if over:
        warnings.warn(f"labels exceed the figure by {over} in: widen YTICK_RESERVE / XTICK_RESERVE_RIGHT")


def legend_handles():
    return [Line2D([], [], color=STYLE[m]["color"], ls=STYLE[m]["ls"], label=m,
                   lw=LINE_W * 1.2 * (EMPHASIS_SCALE if m == EMPHASIS else 1.0),
                   marker=MARKER, ms=MARKER_SIZE, mew=MARKER_EDGE) for m in METHODS]


def fit_legend(ax, legend, curves, grow=1.08, max_iter=30):
    """Raise the y limit (bottom stays 0) until no curve (mean or band top, sampled densely along x) runs
    under the in-panel legend; a no-op when it already fits. Returns the number of steps taken."""
    fig = ax.figure
    for i in range(max_iter):
        fig.canvas.draw()
        box = legend.get_window_extent().padded(2 * fig.dpi / 72)
        hit = False
        for x, y in curves:
            xs = np.linspace(x[0], x[-1], 200)
            pts = ax.transData.transform(np.column_stack([xs, np.interp(xs, x, y)]))
            if np.any((pts[:, 0] >= box.x0) & (pts[:, 0] <= box.x1) & (pts[:, 1] >= box.y0) & (pts[:, 1] <= box.y1)):
                hit = True
                break
        if not hit:
            return i
        ax.set_ylim(0, ax.get_ylim()[1] * grow)
    warnings.warn("legend still overlaps the data: change LEGEND_IN_PANEL")
    return max_iter


def draw_cell(ax, metric, phase):
    """Curves of one (metric, phase) cell; returns [(x, top of mean/band)] for fit_legend."""
    df, spec = timing[phase], METRICS[metric]
    curves = []
    for z, m in enumerate(METHODS):
        sub = df[df["Model"] == LOG_NAME[m]]
        if sub.empty:
            warnings.warn(f"{phase} {m}: no rows, it will not be drawn")
            continue
        st, emph = STYLE[m], m == EMPHASIS
        zorder = 3 + (1 if emph else 0) + z * 0.01
        x = sub["Seq"].to_numpy()
        y, e = sub[spec["col"]].to_numpy() * spec["scale"], sub[spec["err"]].to_numpy() * spec["scale"]
        curves.append((x, y + e if SHOW_STD else y))
        if SHOW_STD:
            ax.fill_between(x, y - e, y + e, color=st["color"], alpha=BAND_ALPHA, lw=0, zorder=zorder - 1)
        ax.plot(x, y, color=st["color"], ls=st["ls"], lw=LINE_W * (EMPHASIS_SCALE if emph else 1.0), zorder=zorder,
                marker=MARKER, ms=MARKER_SIZE, mew=MARKER_EDGE)
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.grid(True, ls="--", alpha=0.5)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    return curves


def clip_top(ax, metric, phase, spec):
    """Cut the y axis at spec["top"] (the outlier curve leaves the panel) and print the outlier's last value."""
    ax.set_ylim(0, spec["top"])
    if spec.get("note"):
        method, text = spec["note"]
        sub = timing[phase][timing[phase]["Model"] == LOG_NAME[method]].sort_values("Seq")
        L, y = int(sub["Seq"].iloc[-1]), sub[METRICS[metric]["col"]].iloc[-1] * METRICS[metric]["scale"]
        ax.text(0.98, 0.94, text.format(y=y, L=L), transform=ax.transAxes, ha="right", va="top",
                color=STYLE[method]["color"], fontsize=matplotlib.rcParams["xtick.labelsize"], zorder=10)


def plot_grid(out_stem=OUT_STEM):
    """Rows = METRICS (time, VRAM), columns = PHASES (rollout, update), one figure."""
    rc, pt = matplotlib.rcParams, 1 / 72
    (w, h), rects = grid_layout()
    fig = plt.figure(figsize=(w, h))
    first_row, last_row, first_col = list(METRICS)[0], list(METRICS)[-1], list(PHASES)[0]
    # every left-column y label at the same x: right edge just left of the fixed y tick-label reserve
    ytick_room = _text_extent(YTICK_RESERVE, rc["ytick.labelsize"])[0] + (rc["ytick.major.size"] + rc["ytick.major.pad"]) * pt
    label_off = ytick_room + rc["axes.labelpad"] * pt
    axes, curves = {}, {}
    for (metric, phase), rect in rects.items():
        x0, y0, pw, ph = rect
        ax = fig.add_axes([x0 / w, y0 / h, pw / w, ph / h])
        curves[(metric, phase)] = draw_cell(ax, metric, phase)
        if metric == first_row:
            ax.set_title(PHASES[phase]["title"])
        if (metric, phase) in YCLIP:
            clip_top(ax, metric, phase, YCLIP[(metric, phase)])
        axes[(metric, phase)] = ax
        if phase == first_col:
            ax.set_ylabel(METRICS[metric]["ylabel"])
            ax.yaxis.set_label_coords(-label_off / pw, 0.5)
        if metric == last_row:
            ax.set_xlabel(XLABEL)
    ticks = set_budget_xticks(axes[(last_row, first_col)], XMAX)   # measured on a labelled (bottom) axis
    for (metric, _phase), ax in axes.items():
        ax.set_xticks(ticks)
        ax.set_xlim(0, XMAX)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _pos: f"{v:g}"))
        if metric != last_row:
            ax.tick_params(axis="x", labelbottom=False)
    if LEGEND_MODE in ("top", "bottom"):                         # one row centered under / over the plot areas
        x_mid = (min(r[0] for r in rects.values()) + max(r[0] + r[2] for r in rects.values())) / 2 / w
        kw = top_legend_kwargs()
        if LEGEND_MODE == "top":
            anchor = (x_mid, 1 - OUTER_PAD / h)
        else:                                                    # leave room below for the drop shadow
            shadow = LEGEND_SHADOW["size"] / 72 if LEGEND_SHADOW else 0
            anchor, kw["loc"] = (x_mid, (OUTER_PAD + shadow) / h), "lower center"
        legend = fig.legend(handles=legend_handles(), bbox_to_anchor=anchor, **kw)
        style_legend_frame(legend)
    elif LEGEND_IN_PANEL:
        ax = axes[LEGEND_CELL]
        kw = dict(frameon=True, fancybox=False, edgecolor=LEGEND_EDGE, facecolor="white", framealpha=1.0,
                  handlelength=1.8, handletextpad=0.4, columnspacing=0.8, borderpad=0.35, labelspacing=0.25,
                  borderaxespad=0.3)
        kw.update(LEGEND_IN_PANEL)
        legend = ax.legend(handles=legend_handles(), **kw)
        legend.set_zorder(10)
        style_legend_frame(legend)
        if fit_legend(ax, legend, curves[LEGEND_CELL]):
            print(f"{LEGEND_CELL}: y limit raised to {ax.get_ylim()[1]:.3g} to clear the legend")
    check_fits(fig)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    stem = FIG_DIR / out_stem
    fig.savefig(stem.with_suffix(".pdf"))
    fig.savefig(stem.with_suffix(".png"), dpi=300)
    pw, ph = next(iter(rects.values()))[2:]
    print(f"saved {stem.with_suffix('.pdf')} ({w:.2f} x {h:.2f} in, each plot area {pw:.2f} x {ph:.2f} in)")
    return fig, axes

In [5]:
# rows: time (top), peak VRAM (bottom); columns: rollout (left), update (right); hidden_size=128
fig, axes = plot_grid()
plt.show()

saved figures/timing/compute_cost.pdf (5.50 x 2.35 in, each plot area 2.34 x 0.75 in)


## Std bound (for the caption)

The logs round the time to 0.1 ms (s with 4 decimals) and memory to 0.01 MB, so many stds read exactly 0.
A rounded value is still within half a digit of the true one, so `(std + u) / (mean - u)` (u = half a digit) is a
guaranteed upper bound on std / mean, converted to the sample std (ddof = 1). It is loosest for the shortest,
fastest points (e.g. a 1.6 ms update: rounding alone allows 3%).

In [6]:
def std_bound(metric, phase, sample_std=True):
    """Upper bound on std / mean per point, valid although the logs round (a logged 0.0000 s std only means
    std < 0.00005 s): (std_logged + u) / (mean_logged - u), u = half of the last logged digit. With
    sample_std=True the logged population std (ddof=0) is converted to the sample std (x sqrt(n / (n - 1)))."""
    df, spec = timing[phase], METRICS[metric]
    u = LOG_HALF_UNIT[metric]
    k = np.sqrt(N_TRIALS / (N_TRIALS - 1)) if sample_std else 1.0
    return df.assign(bound=k * (df[spec["err"]] + u) / (df[spec["col"]] - u))


rows = []
for metric in METRICS:
    for phase in PHASES:
        b = std_bound(metric, phase)
        worst = b.loc[b["bound"].idxmax()]
        rows.append(dict(metric=metric, phase=phase, max_bound=b["bound"].max(), median_bound=b["bound"].median(),
                         n_over_2pct=int((b["bound"] > 0.02).sum()), n_points=len(b),
                         worst=f"{worst['Model']} L={worst['Seq']}"))
bounds = pd.DataFrame(rows).set_index(["metric", "phase"])
with pd.option_context("display.float_format", "{:.2%}".format):
    display(bounds)

t, v = bounds.loc["time"], bounds.loc["vram"]
print("Caption: mean over {n} timed runs (after 3 warm-up runs); the standard deviation is at most {r:.0f}% (rollout) "
      "and {u:.0f}% (update) of the mean at every point (median {m:.0f}%), and below {vm:.1f}% for peak memory, "
      "so it is not drawn.".format(n=N_TRIALS, r=np.ceil(100 * t.loc["rollout", "max_bound"]),
                                   u=np.ceil(100 * t.loc["update", "max_bound"]),
                                   m=np.ceil(100 * t["median_bound"].max()),
                                   vm=np.ceil(1000 * v["max_bound"].max()) / 10))

max_bound  median_bound  n_over_2pct  n_points           worst
metric phase                                                                  
time   rollout      6.66%         1.74%           21        50     mamba L=200
       update      13.92%         2.40%           27        50       gpt L=100
vram   rollout      0.06%         0.03%            0        50      mate L=100
       update       0.48%         0.00%            0        50  splagger L=100

Caption: mean over 10 timed runs (after 3 warm-up runs); the standard deviation is at most 7% (rollout) and 14% (update) of the mean at every point (median 3%), and below 0.5% for peak memory, so it is not drawn.


## Numbers at the longest sequence (ratio to MATE)

In [7]:
rows = []
for phase, df in timing.items():
    for m in METHODS:
        sub = df[(df["Model"] == LOG_NAME[m]) & (df["Seq"] == df["Seq"].max())]
        if len(sub):
            r = sub.iloc[0]
            rows.append(dict(phase=phase, method=m, params=r["Params"], time_ms=r["Time"] * 1e3, vram_gb=r["VRAM"] / 1024))
table = pd.DataFrame(rows)
mate = table[table["method"] == "MATE"].set_index("phase")
table["time_x_mate"] = table["time_ms"] / table["phase"].map(mate["time_ms"])
table["vram_x_mate"] = table["vram_gb"] / table["phase"].map(mate["vram_gb"])
with pd.option_context("display.float_format", "{:.3g}".format):
    display(table.set_index(["phase", "method"]))

params  time_ms  vram_gb  time_x_mate  vram_x_mate
phase   method                                                      
rollout MATE       18177      176  0.00815            1            1
        GPT-2     200064      525    0.225         2.97         27.6
        LSTM      133632      198   0.0178         1.12         2.19
        SplAgger  100608      286   0.0173         1.62         2.12
        Mamba     118528      373    0.011         2.12         1.34
update  MATE       18177      2.3    0.237            1            1
        GPT-2     200064       14     1.03         6.09         4.35
        LSTM      133632        8     1.02         3.48         4.32
        SplAgger  100608      5.9    0.993         2.57         4.19
        Mamba     118528     12.5    0.903         5.43         3.81

## LaTeX

```latex
\begin{figure}[t]
  \centering
  \includegraphics{figures/timing/compute_cost.pdf}
  \caption{Wall-clock time (top) and peak GPU memory (bottom) of a rollout (left; $L$ sequential steps,
    batch 64) and of one update (right; forward + backward on a batch of 64 length-$L$ sequences) vs sequence
    length $L$ (hidden size 128, one layer). Mean over 10 timed runs; the standard deviation is at most 7\% (rollout)
    and 14\% (update) of the mean at every point (median $\le$ 3\%) and below 0.5\% for memory, so it is not drawn.
    GPT-2's rollout memory leaves the panel; its value at $L=1000$ is printed.}
  \label{fig:compute-time}
\end{figure}
```

No `width=`: the figure is designed at its final size (font sizes from `paper_style.py`).